# CROSS replication on Kaggle (brain-tumour MRI, held-out-source protocol)

**Inputs (Add Input):** (1) the dataset `masoudnickparvar/brain-tumor-mri-dataset`; (2) your private dataset made
from `cross_bundle.zip`. **Settings:** GPU (T4 or P100), Internet **on** (timm ImageNet weights).
Run with *Save Version → Save & Run All (Commit)*. Finished runs are skipped, so a re-run resumes.
Outputs: `/kaggle/working/cross_kaggle/{results.csv, preds/}` → download and copy to `results/cross_kaggle/`.

In [ ]:
import glob, os, shutil, subprocess
def find(rel):
    # fixed-depth search (a recursive glob over the mounted inputs takes minutes)
    for depth in range(0, 5):
        hits = sorted(glob.glob("/kaggle/input/" + "*/" * depth + rel))
        if hits:
            return hits[0]
    raise FileNotFoundError(rel)
bundle = find("results/audit/grouped_folds.csv")[: -len("results/audit/grouped_folds.csv")].rstrip("/")
data = find("Training/glioma")[: -len("Training/glioma")].rstrip("/")   # the folder that holds Training/ and Testing/
assert os.path.isdir(f"{data}/Testing"), data
P = "/kaggle/working/proj"
if not os.path.exists(P):
    shutil.copytree(bundle, P)
os.makedirs(f"{P}/data", exist_ok=True)
if not os.path.exists(f"{P}/data/brain-tumor-mri-dataset"):
    os.symlink(data, f"{P}/data/brain-tumor-mri-dataset")
print("bundle:", bundle, "| data:", data)
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import sys
env = dict(os.environ, CROSS_OUT="/kaggle/working/cross_kaggle", PYTHONUNBUFFERED="1")
def go(arms, seeds):
    cmd = [sys.executable, "code/cross_pilot.py", "--arms", arms, "--folds", "0,1,2,3,4", "--seeds", seeds]
    p = subprocess.Popen(cmd, cwd=P, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0
# QUICK_TEST: one 1-epoch run to check the setup (writes to a separate folder)
QUICK_TEST = False
if QUICK_TEST:
    env["CROSS_OUT"] = "/kaggle/working/cross_quick"
    subprocess.run([sys.executable, "code/cross_pilot.py", "--arms", "cross_p_r25", "--folds", "0", "--seeds", "1", "--epochs", "1"], cwd=P, env=env, check=True)
else:
    go("erm,cross_p,cross_p_r25", "43,44")   # 30 runs: replication of the pilot, and the lower transplant rate
    go("erm,cross_p_r25", "42")              # 10 runs: Kaggle ERM at the pilot seed, and the lower rate at seed 42

In [ ]:
shutil.make_archive("/kaggle/working/cross_kaggle_results", "zip", "/kaggle/working/cross_kaggle")
print(open("/kaggle/working/cross_kaggle/results.csv").read())